THIS NOTEBOOK IS BEING DEVELOP TO SETUP KAFKA PROGRAM


In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "uat", "prod"], "Environment")
env = dbutils.widgets.get("env")
print(f"Running in Environment: {env}")

In [0]:
# Load environment config

import os
import json

# Build the config file path relative to the notebook location (src/ingestion/)
config_relative_path = os.path.join("..", "..", "config", f"{env}.json")
config_path = os.path.realpath(config_relative_path)

print(f"Resolved config path: {config_path}")

# Check the file exists
if not os.path.isfile(config_path):
    raise FileNotFoundError(
        f"Config file for env '{env}' not found at {config_path}. "
        f"Expected relative path '{config_relative_path}' from the notebook directory."
    )

# Load the JSON config
with open(config_path, "r") as f:
    config = json.load(f)

print(f"Successfully loaded config from: {config_path}")

# Validate that all required keys are present
REQUIRED_KEYS = [
    "CATALOG",
    "SCHEMA",
    "KAFKA_TABLE",
    "CHECKPOINT_VOLUME",
    "CHECKPOINT_FOLDER",
    "KAFKA_TOPIC",
    "STARTING_OFFSETS",
    "SECRET_SCOPE"
]

missing_keys = [k for k in REQUIRED_KEYS if k not in config]
if missing_keys:
    raise KeyError(
        f"Config file '{config_path}' is missing required keys: {missing_keys}. "
        f"Please check for typos. Required keys are: {REQUIRED_KEYS}"
    )

# Print the loaded configuration
print(f"\n{'='*50}")
print(f"Environment : {env}")
print(f"Config file : {config_path}")
print(f"{'='*50}")
for key in REQUIRED_KEYS:
    print(f"  {key:35s} : {config[key]}")
print(f"{'='*50}")

In [0]:
# Build full names from config parts (single place for all environment-specific names)
catalog = config["CATALOG"]
schema = config["SCHEMA"]
bronze_table = config["KAFKA_TABLE"]
checkpoint_volume = config["CHECKPOINT_VOLUME"]
checkpoint_folder = config["CHECKPOINT_FOLDER"]

# Fully qualified table name; backticks keep names like 161pmprojectcatalog safe
bronze_table_fqn = f"`{catalog}`.`{schema}`.`{bronze_table}`"

# Checkpoint path in the Unity Catalog volume (a file path, so no backticks)
checkpoint_path = f"/Volumes/{catalog}/{schema}/{checkpoint_volume}/{checkpoint_folder}"

print(f"Bronze table    : {bronze_table_fqn}")
print(f"Checkpoint path : {checkpoint_path}")

In [0]:

from pyspark.sql import functions as F
SECRET_SCOPE=config["SECRET_SCOPE"]
bootstrap_servers=dbutils.secrets.get(SECRET_SCOPE, "kafka-bootstrap-servers")
KAFKA_TOPIC = config["KAFKA_TOPIC"]
streaming_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", KAFKA_TOPIC)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
        f'username="{dbutils.secrets.get(SECRET_SCOPE, "kafka-api-key")}" '
        f'password="{dbutils.secrets.get(SECRET_SCOPE, "kafka-api-secret")}";',
    )
    .option("startingOffsets", config["STARTING_OFFSETS"])
    .load()
)



In [0]:
from pyspark.sql.functions import col
parsed_streaming_df =streaming_df.select(col("key").cast("string"),col("value").cast("string"),col("topic"),col("partition"),col("offset"),col("timestamp"),col("timestampType") )


In [0]:
# Write raw Kafka messages to the bronze table and wait until the available-now run finishes
streaming_query = (
    parsed_streaming_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table_fqn)
)
print(f"Query started with query id: {streaming_query.id}")

# Block until all available messages are written; raises an error here if the stream fails
streaming_query.awaitTermination()



In [0]:
# Verify the bronze table using the name built from config
bronze_count = spark.table(bronze_table_fqn).count()
print(f"Rows in {bronze_table_fqn}: {bronze_count}")